# The small run, stage by stage

One shot of the small example (d=3, 12 rounds, 3 sliding windows) walked
through **every stage of the pipeline**, in path order. Every stage shows
three things: what goes **in**, what comes **out**, and the **timestamps**.

```
QPU -> QC link -> controller (pulses to binary) -> syndrome packing
    -> C2B link -> Buffer 0 -> window manager -> CWD link -> decoder memory
    -> decoder engine (fetch -> algorithm -> release) -> DD handoff
    -> WDO link -> Pauli frame commit
```

The run uses p = 0.001 and seed 8, chosen so a real defect flows through:
one detector fires, the logical observable really flips, and the decoder
catches it. The stage costs are preset cards, so every timestamp is still
exactly the zero-noise hand arithmetic of README section 1; the last cell
checks that, cell for cell.

This notebook is a viewing layer: the same run works from the shell with
`analytic_small_run.py` and `simulated_small_run.py` in this folder.

In [1]:
# Make the repository root and this folder importable, wherever jupyter started.
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "guide" / "walkthrough"))
sys.path.insert(0, str(repo_root))
print(repo_root)

/scratch/gpfs/MARTONOSI/sk2415/qlx-qec-sandbox/decsim


## The configuration: one yaml names every cost

Round period 1.0 µs. QC 0.15, C2B 0.10, CWD 2.0, DD 0.5, WDO 1.0 (all
propagation only, unbounded bandwidth). Controller processing and packing
0. Engine 250 MHz (0.004 µs per cycle), algorithm card 0.028, frame commit
0.004.

In [2]:
from experiments.baseline.baseline_closed_loop import build_run, load_config

CONFIG_PATH = repo_root / "experiments/validation/analytic_oracle_d3_r12.yaml"
print(CONFIG_PATH.read_text())

# Deterministic analytical-oracle case.
#
# Purpose:
#   Verify the exact baseline event ordering and timing, not performance or LER.
#   This case intentionally has:
#     - zero physical noise,
#     - one fixed decoder latency,
#     - one decoder unit,
#     - unbounded link bandwidth,
#     - only 12 QEC rounds, which produce exactly 3 sliding windows.
#
# Expected analytical results are provided beside this file.

code_task: surface_code:rotated_memory_z
distance: 3
rounds_per_shot: 12

windowing:
  scheme: sliding
  commit_rounds: 3
  buffer_rounds: 3

sweep:
  # p = 0 exactly cannot run: a noiseless circuit has an empty detector error
  # model and no matching graph. 1.0e-9 samples zero defects on every shot and
  # every timing tick is identical to p = 0 (preset stage costs, payload sizes
  # independent of p).
  - physical_error_probability: [1.0e-9]
    round_period_us: [1.0]
    algorithm_latency_us: [0.028]
    shots: 1

controller:
  t_binary_availability_us: 0.0
  t_pack

## Run the shot

`build_run` assembles the path from the yaml; `spec.build()` replays it in
the event loop. Every later cell only *reads the records* this run left
behind: link transfers, window timestamps, engine stage records, Pauli
frame records.

In [3]:
from decsim.config import TICKS_PER_US

config = load_config(CONFIG_PATH)
spec, decoder_engine = build_run(config,
                                 physical_error_probability=0.001,
                                 round_period_us=1.0,
                                 algorithm_latency_us=0.028,
                                 seed=8)
completed = spec.build()

transfers = completed.result.link_traffic["transfers"]
windows = {window_id: window
           for (_, window_id), window in sorted(completed.window_manager.windows.items())}
frame_records = {record.window_key[1]: record
                 for record in completed.pauli_frame.snapshot().records}


def us(ticks):
    return ticks / TICKS_PER_US


def link(path):
    """round or window id -> this path's transfer record."""
    key = "round_lo" if path in ("qc", "c2b") else "window_id"
    return {t["attribution"][key]: t for t in transfers if t["path"] == path}


qc, c2b = link("qc"), link("c2b")
cwd, dd, wdo = link("cwd"), link("dd"), link("wdo")
print(completed.result.terminal_status)

complete


## Stage 1: the QPU

**IN**: the Stim circuit below, executed one round every 1.000 µs
(seed 8 fixes the noise sample).
**OUT**: one readout fragment per round: that round's detector bits.
Round r finishes, and its fragment leaves the QPU, at r x 1.000 µs.

Watch round 11: one bit is set. A physical error happened between rounds
10 and 11 and exactly one stabilizer comparison caught it.

In [4]:
operation = spec.ops[0]
print(operation.circuit)

QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(1, 3) 8
QUBIT_COORDS(2, 2) 9
QUBIT_COORDS(3, 3) 10
QUBIT_COORDS(4, 2) 11
QUBIT_COORDS(5, 3) 12
QUBIT_COORDS(6, 2) 13
QUBIT_COORDS(0, 4) 14
QUBIT_COORDS(1, 5) 15
QUBIT_COORDS(2, 4) 16
QUBIT_COORDS(3, 5) 17
QUBIT_COORDS(4, 4) 18
QUBIT_COORDS(5, 5) 19
QUBIT_COORDS(4, 6) 25
R 1 3 5 8 10 12 15 17 19
X_ERROR(0.001) 1 3 5 8 10 12 15 17 19
R 2 9 11 13 14 16 18 25
X_ERROR(0.001) 2 9 11 13 14 16 18 25
TICK
DEPOLARIZE1(0.001) 1 3 5 8 10 12 15 17 19
H 2 11 16 25
DEPOLARIZE1(0.001) 2 11 16 25
TICK
CX 2 3 16 17 11 12 15 14 10 9 19 18
DEPOLARIZE2(0.001) 2 3 16 17 11 12 15 14 10 9 19 18
TICK
CX 2 1 16 15 11 10 8 14 3 9 12 18
DEPOLARIZE2(0.001) 2 1 16 15 11 10 8 14 3 9 12 18
TICK
CX 16 10 11 5 25 19 8 9 17 18 12 13
DEPOLARIZE2(0.001) 16 10 11 5 25 19 8 9 17 18 12 13
TICK
CX 16 8 11 3 25 17 1 9 10 18 5 13
DEPOLARIZE2(0.001) 16 8 11 3 25 17 1 9 10 18 5 13
TICK
H 2 11 16 25
DEPOLARIZE1(0.001) 2 11 16 25
TICK
X

In [5]:
model = completed.qpu.model
round_bits = {}
for round_index in sorted(qc):
    fragment = model.round_payloads(operation, round_index)[0]
    round_bits[round_index] = "".join(str(int(bit)) for bit in fragment.bits)

print("round | leaves QPU (µs) | detector bits | fired")
for round_index, bits in round_bits.items():
    print(f"{round_index:5} | {us(qc[round_index]['send_ticks']):15.3f} "
          f"| {bits:>13} | {bits.count('1')}")

round | leaves QPU (µs) | detector bits | fired
    1 |           1.000 |          0000 | 0
    2 |           2.000 |      00000000 | 0
    3 |           3.000 |      00000000 | 0
    4 |           4.000 |      00000000 | 0
    5 |           5.000 |      00000000 | 0
    6 |           6.000 |      00000000 | 0
    7 |           7.000 |      00000000 | 0
    8 |           8.000 |      00000000 | 0
    9 |           9.000 |      00000000 | 0
   10 |          10.000 |      00000000 | 0
   11 |          11.000 |      00010000 | 1
   12 |          12.000 |  000000000000 | 0


## Stage 2: the QC link (QPU -> controller)

**IN**: the fragment, at its send tick.
**OUT**: the same fragment at the controller, 0.150 µs later (propagation
only; the link is unbounded, so serialization and queueing are zero).

In [6]:
print("round | bits | sent (µs) | at controller (µs) | delay (µs)")
for round_index in sorted(qc):
    transfer = qc[round_index]
    print(f"{round_index:5} | {transfer['payload_bits']:4} "
          f"| {us(transfer['send_ticks']):9.3f} | {us(transfer['delivery_ticks']):18.3f} "
          f"| {us(transfer['delivery_ticks'] - transfer['send_ticks']):10.3f}")

round | bits | sent (µs) | at controller (µs) | delay (µs)
    1 |    4 |     1.000 |              1.150 |      0.150
    2 |    8 |     2.000 |              2.150 |      0.150
    3 |    8 |     3.000 |              3.150 |      0.150
    4 |    8 |     4.000 |              4.150 |      0.150
    5 |    8 |     5.000 |              5.150 |      0.150
    6 |    8 |     6.000 |              6.150 |      0.150
    7 |    8 |     7.000 |              7.150 |      0.150
    8 |    8 |     8.000 |              8.150 |      0.150
    9 |    8 |     9.000 |              9.150 |      0.150
   10 |    8 |    10.000 |             10.150 |      0.150
   11 |    8 |    11.000 |             11.150 |      0.150
   12 |   12 |    12.000 |             12.150 |      0.150


## Stage 3: controller processing (pulses to binary)

**IN**: the delivered fragment.
**OUT**: the same bits, declared binary-available after
`t_binary_availability_us`. This yaml sets it to 0.0 (readout
classification is priced inside the round), so availability = delivery.

In [7]:
t_binary_us = config["controller"]["t_binary_availability_us"]
print(f"t_binary_availability_us = {t_binary_us}")
print("round | at controller (µs) | binary available (µs)")
for round_index in sorted(qc):
    delivered_us = us(qc[round_index]["delivery_ticks"])
    print(f"{round_index:5} | {delivered_us:18.3f} | {delivered_us + t_binary_us:21.3f}")

t_binary_availability_us = 0.0
round | at controller (µs) | binary available (µs)
    1 |              1.150 |                 1.150
    2 |              2.150 |                 2.150
    3 |              3.150 |                 3.150
    4 |              4.150 |                 4.150
    5 |              5.150 |                 5.150
    6 |              6.150 |                 6.150
    7 |              7.150 |                 7.150
    8 |              8.150 |                 8.150
    9 |              9.150 |                 9.150
   10 |             10.150 |                10.150
   11 |             11.150 |                11.150
   12 |             12.150 |                12.150


## Stage 4: syndrome packing

**IN**: all fragments of round r (this device emits exactly one per round,
so there is never a wait for missing fragments).
**OUT**: one `SyndromeRoundPacket` whose size is the fragments' bit sum,
ready after `t_pack_us` (0.0 here, packing one fragment is a no-op). The
packet leaves for Buffer 0 immediately: the C2B send tick equals the packed
tick.

In [8]:
t_pack_us = config["controller"]["t_pack_us"]
print(f"t_pack_us = {t_pack_us}")
print("round | fragments in | bits in | packet bits out | packed at (µs) | leaves controller (µs)")
for round_index in sorted(qc):
    fragment = model.round_payloads(operation, round_index)[0]
    packed_us = us(qc[round_index]["delivery_ticks"]) + t_binary_us + t_pack_us
    print(f"{round_index:5} | {fragment.n_fragments:12} | {fragment.size_bits:7} "
          f"| {c2b[round_index]['payload_bits']:15} | {packed_us:14.3f} "
          f"| {us(c2b[round_index]['send_ticks']):22.3f}")

t_pack_us = 0.0
round | fragments in | bits in | packet bits out | packed at (µs) | leaves controller (µs)
    1 |            1 |       4 |               4 |          1.150 |                  1.150
    2 |            1 |       8 |               8 |          2.150 |                  2.150
    3 |            1 |       8 |               8 |          3.150 |                  3.150
    4 |            1 |       8 |               8 |          4.150 |                  4.150
    5 |            1 |       8 |               8 |          5.150 |                  5.150
    6 |            1 |       8 |               8 |          6.150 |                  6.150
    7 |            1 |       8 |               8 |          7.150 |                  7.150
    8 |            1 |       8 |               8 |          8.150 |                  8.150
    9 |            1 |       8 |               8 |          9.150 |                  9.150
   10 |            1 |       8 |               8 |         10.150 |       

## Stage 5: the C2B link (controller -> Buffer 0)

**IN**: the packet, at its send tick.
**OUT**: the round **published** in Buffer 0, 0.100 µs later. Publication
is the moment the window manager can see the round; the running count shows
Buffer 0 filling up. Rounds land every 1.000 µs, a constant 0.250 µs after
they left the QPU: links shift the phase, never the rate.

In [9]:
print("round | sent (µs) | published in Buffer 0 (µs) | rounds in Buffer 0")
for round_index in sorted(c2b):
    transfer = c2b[round_index]
    print(f"{round_index:5} | {us(transfer['send_ticks']):9.3f} "
          f"| {us(transfer['delivery_ticks']):26.3f} | {round_index:18}")

round | sent (µs) | published in Buffer 0 (µs) | rounds in Buffer 0
    1 |     1.150 |                      1.250 |                  1
    2 |     2.150 |                      2.250 |                  2
    3 |     3.150 |                      3.250 |                  3
    4 |     4.150 |                      4.250 |                  4
    5 |     5.150 |                      5.250 |                  5
    6 |     6.150 |                      6.250 |                  6
    7 |     7.150 |                      7.250 |                  7
    8 |     8.150 |                      8.250 |                  8
    9 |     9.150 |                      9.250 |                  9
   10 |    10.150 |                     10.250 |                 10
   11 |    11.150 |                     11.250 |                 11
   12 |    12.150 |                     12.250 |                 12


## Stage 6: the window manager (rounds -> decode jobs)

**IN**: published rounds.
**OUT**: decode jobs. A window is **ready** when the last round it reads is
published; it is **queued** once its dependency (the previous window's DD
boundary handoff) has arrived; it **dispatches** when a decoder unit is
free. Window 0 has no dependency; windows 1 and 2 wait 0.556 µs for theirs,
but their own data arrives even later, so nothing blocks here.

In [10]:
print("window | reads | commits | first round in (µs) | ready (µs) | depends on | queued (µs) | dispatch (µs)")
for window_id, window in windows.items():
    dependencies = ",".join(str(dep[1]) for dep in window.deps) or "-"
    read_hi = min(window.buffer_hi, config["rounds_per_shot"])
    reads = f"{window.start_round}..{read_hi}"
    commits = f"{window.commit_lo}..{window.commit_hi}"
    print(f"{window_id:6} | {reads:>5} | {commits:>7} "
          f"| {us(window.t_first_round):19.3f} | {us(window.t_data_complete):10.3f} "
          f"| {dependencies:>10} | {us(window.t_queued):11.3f} | {us(window.t_dispatch):12.3f}")

window | reads | commits | first round in (µs) | ready (µs) | depends on | queued (µs) | dispatch (µs)
     0 |  1..6 |    1..3 |               1.250 |      6.250 |          - |       6.250 |        6.250
     1 |  4..9 |    4..6 |               4.250 |      9.250 |          0 |       9.250 |        9.250
     2 | 7..12 |   7..12 |               7.250 |     12.250 |          1 |      12.250 |       12.250


## Stage 7: the CWD link and decoder memory

**IN**: the dispatched window's rounds out of Buffer 0 (the bit sum of the
6 rounds it reads).
**OUT**: those bits sitting in the decoder unit's input memory, 2.000 µs
later.

In [11]:
print("window | window payload bits | leaves Buffer 0 (µs) | in decoder memory (µs)")
for window_id in windows:
    transfer = cwd[window_id]
    print(f"{window_id:6} | {transfer['payload_bits']:19} "
          f"| {us(transfer['send_ticks']):20.3f} | {us(transfer['delivery_ticks']):22.3f}")

window | window payload bits | leaves Buffer 0 (µs) | in decoder memory (µs)
     0 |                  44 |                6.250 |                  8.250
     1 |                  48 |                9.250 |                 11.250
     2 |                  52 |               12.250 |                 14.250


## Stage 8: the decoder engine (fetch -> algorithm -> release)

**IN**: the window's bits in decoder memory.
**OUT**: this window's correction (its logical observable estimate).
Fetch reads the 6 rounds out of memory (6 cycles = 0.024 µs), the algorithm
card charges 0.028 µs, release writes the correction out (1 cycle =
0.004 µs). Window 2's correction is 1: it saw the defect and corrected the
flip.

In [12]:
print("window | fetch (µs) | algorithm (µs) | release (µs) | decode done (µs) | correction")
for window_id, window in windows.items():
    stages = {record.stage: record
              for record in decoder_engine.stage_records_for(operation.id, window_id)}
    def span(name):
        record = stages[name]
        return f"{us(record.start_ticks):.3f}-{us(record.end_ticks):.3f}"
    correction = frame_records[window_id].logical_observables
    print(f"{window_id:6} | {span('fetch'):>13} | {span('algorithm'):>14} | {span('release'):>13} "
          f"| {us(window.t_done):16.3f} | {correction}")

window | fetch (µs) | algorithm (µs) | release (µs) | decode done (µs) | correction
     0 |   8.250-8.274 |    8.274-8.302 |   8.302-8.306 |            8.306 | (0,)
     1 | 11.250-11.274 |  11.274-11.302 | 11.302-11.306 |           11.306 | (0,)
     2 | 14.250-14.274 |  14.274-14.302 | 14.302-14.306 |           14.306 | (1,)


## Stage 9: the DD handoff (decoder -> next window's decoder)

**IN**: the decoded window's boundary data, at decode done.
**OUT**: delivered to the next window's decode 0.500 µs later; that
delivery is exactly the dependency arrival that gated stage 6. The last
window has no successor and sends nothing.

In [13]:
print("window | sent (µs) | delivered to next window (µs)")
for window_id in windows:
    if window_id in dd:
        transfer = dd[window_id]
        print(f"{window_id:6} | {us(transfer['send_ticks']):9.3f} "
              f"| {us(transfer['delivery_ticks']):29.3f}")
    else:
        print(f"{window_id:6} | (last window, no handoff)")

window | sent (µs) | delivered to next window (µs)
     0 |     8.306 |                         8.806
     1 |    11.306 |                        11.806
     2 | (last window, no handoff)


## Stage 10: the WDO link (decoder -> Pauli frame)

**IN**: the correction, leaving the decoder at decode done.
**OUT**: the correction at the Pauli frame, 1.000 µs later.

In [14]:
print("window | leaves decoder (µs) | at Pauli frame (µs)")
for window_id in windows:
    transfer = wdo[window_id]
    print(f"{window_id:6} | {us(transfer['send_ticks']):19.3f} "
          f"| {us(transfer['delivery_ticks']):19.3f}")

window | leaves decoder (µs) | at Pauli frame (µs)
     0 |               8.306 |               9.306
     1 |              11.306 |              12.306
     2 |              14.306 |              15.306


## Stage 11: the Pauli frame commit

**IN**: the delivered correction.
**OUT**: the frame updated 0.004 µs later. XOR of the committed
corrections = the loop's logical prediction. The truth comes from the
QPU's own sampled observable flip: they agree, so the shot is decoded
correctly even though the observable really flipped.

In [15]:
print("window | accepted (µs) | committed (µs) | correction")
for window_id, record in sorted(frame_records.items()):
    print(f"{window_id:6} | {us(record.accepted_ticks):13.3f} "
          f"| {us(record.committed_ticks):14.3f} | {record.logical_observables}")

operation_result = completed.result.operation_results[0]
print(f"\nloop prediction: {operation_result.logical_observables}")
print(f"observable truth: {operation_result.observable_truth}")
print(f"logical failure: {operation_result.logical_observables != tuple(operation_result.observable_truth)}")

window | accepted (µs) | committed (µs) | correction
     0 |         9.306 |          9.310 | (0,)
     1 |        12.306 |         12.310 | (0,)
     2 |        15.306 |         15.310 | (1,)

loop prediction: (1,)
observable truth: (1,)
logical failure: False


## The end-to-end table, checked against the hand arithmetic

The same run condensed to one row per window, compared cell for cell
against README section 1's closed-form answer key. The defect changed the
*data* (window 2's correction), but not one *timestamp*: stage costs are
preset cards, so timing is independent of the noise.

In [16]:
from analytic_small_run import COLUMNS, analytic_timeline, print_table
from simulated_small_run import differences

simulated_rows = []
for window_id, window in windows.items():
    read_hi = min(window.buffer_hi, config["rounds_per_shot"])
    record = frame_records[window_id]
    simulated_rows.append({
        "window_id": window_id, "read_lo": window.start_round, "read_hi": read_hi,
        "commit_lo": window.commit_lo, "commit_hi": window.commit_hi,
        "buffer0_ready_us": us(window.t_data_complete),
        "queued_us": us(window.t_queued),
        "dispatch_us": us(window.t_dispatch),
        "decode_done_us": us(window.t_done),
        "dd_delivery_us": us(dd[window_id]["delivery_ticks"]) if window_id in dd else None,
        "frame_commit_us": us(record.committed_ticks),
        "buffer0_ready_to_frame_us": us(record.committed_ticks - window.t_data_complete),
    })
print_table(simulated_rows)

disagreements = differences(analytic_timeline(config), simulated_rows)
assert not disagreements, disagreements
print(f"\nMATCH: all {len(simulated_rows) * len(COLUMNS)} cells agree to the tick.")

| window_id | read_lo | read_hi | commit_lo | commit_hi | buffer0_ready_us | queued_us | dispatch_us | decode_done_us | dd_delivery_us | frame_commit_us | buffer0_ready_to_frame_us |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 0 | 1 | 6 | 1 | 3 | 6.250 | 6.250 | 6.250 | 8.306 | 8.806 | 9.310 | 3.060 |
| 1 | 4 | 9 | 4 | 6 | 9.250 | 9.250 | 9.250 | 11.306 | 11.806 | 12.310 | 3.060 |
| 2 | 7 | 12 | 7 | 12 | 12.250 | 12.250 | 12.250 | 14.306 |  | 15.310 | 3.060 |

MATCH: all 36 cells agree to the tick.


## Try a knob

Change one number in the yaml (`cwd.latency_us`, `round_period_us`, the
seed above) and rerun all cells: every stage table shifts exactly as the
five formulas of README section 1 predict, and the final check must still
say MATCH (unless you changed a cost, in which case redo the arithmetic
first and watch it agree again). The frequency sweep and figures are README
section 5.